# InvoiceAI — Evaluation on SROIE

Measures how accurate the pipeline is on real receipts with known answers.

**Plan (about 60–75 minutes on a T4):**
1. Compare prompt **v1** and **v2** on 40 **train** receipts (≈ 20 min).
2. Run the better prompt on 100 **test** receipts (≈ 30 min). These are the numbers for the README.

Why two splits? We choose the prompt on *train* and report on *test*, so the final numbers are honest (we did not tune on them).

**Before you start:** `Runtime → Change runtime type → T4 GPU`. Keep this browser tab open while it runs.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Get the code

In [ ]:
GITHUB_USER = "YOUR_GITHUB_USERNAME"  # <- change this
REPO_DIR = "/content/invoice-ai"

import os
if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone https://github.com/{GITHUB_USER}/invoice-ai.git {REPO_DIR}
!ls {REPO_DIR}/eval

## 3. Install requirements

If Colab asks you to **restart the session**, click Restart and continue with step 4.

In [ ]:
!pip install -q -r /content/invoice-ai/requirements.txt

## 4. Where to save results

Colab deletes files when the runtime disconnects. If you set `USE_DRIVE = True`, results are saved to your Google Drive,
and an interrupted run continues where it stopped when you run it again.

In [ ]:
USE_DRIVE = False  # <- set True to save to Google Drive (recommended for long runs)

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = "/content/drive/MyDrive/invoice-ai-eval"
else:
    OUTPUT_ROOT = "/content/invoice-ai/eval"
print("Results will be saved in:", OUTPUT_ROOT)

## 5. Load the models

In [ ]:
import os, sys, json, time, logging
os.chdir("/content/invoice-ai")
sys.path.insert(0, "/content/invoice-ai")

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", stream=sys.stdout, force=True)
logging.getLogger("app").setLevel(logging.WARNING)  # keep the output short during evaluation

from app.pipeline import get_pipeline
from eval.evaluate import load_sroie, run_evaluation, worst_examples, to_markdown

pipeline = get_pipeline()
start = time.time()
pipeline.load()
print(f"Models loaded in {time.time() - start:.1f} s")

## 6. Check the dataset format

Always look at the data before trusting a number.

In [ ]:
from datasets import load_dataset
from IPython.display import display

ds = load_dataset("jsdnrs/ICDAR2019-SROIE", split="test")
print(ds)
print("\nColumns:", ds.features)

example = ds[0]
print("\nKey:", example["key"])
print("Labels:", json.dumps(example["entities"], indent=2))
preview = example["image"].copy()
preview.thumbnail((400, 800))
display(preview)

## 7. Quick test on 3 receipts

Makes sure everything works before the long runs. `ok` = correct, `x` = wrong, `-` = no label.

In [ ]:
report = run_evaluation(pipeline, load_sroie("train", limit=3), output_dir=f"{OUTPUT_ROOT}/runs/smoke_test",
                        prompt_version="v1", split="train", resume=False)
for field, m in report["summary"]["fields"].items():
    print(f"{field:15} exact: {m['exact_match']}")

## 8. Compare prompt v1 vs v2 (train split, 40 receipts each)

≈ 20 minutes. v2 has clearer rules for vendor name, address and total.

In [ ]:
import pandas as pd

N_COMPARE = 40
comparison = {}
for version in ["v1", "v2"]:
    print(f"\n===== Prompt {version} =====")
    report = run_evaluation(pipeline, load_sroie("train", limit=N_COMPARE), output_dir=f"{OUTPUT_ROOT}/runs/train_{version}",
                            prompt_version=version, split="train")
    comparison[version] = report["summary"]

rows = []
for field in comparison["v1"]["fields"]:
    row = {"field": field}
    for version, summary in comparison.items():
        row[f"{version} exact"] = summary["fields"][field]["exact_match"]
        row[f"{version} fuzzy≥90"] = summary["fields"][field]["fuzzy_match"]
    rows.append(row)
table = pd.DataFrame(rows)
display(table.style.format({c: "{:.1%}" for c in table.columns if c != "field"}))

def average_exact(summary):
    values = [m["exact_match"] for m in summary["fields"].values() if m["exact_match"] is not None]
    return sum(values) / len(values)

BEST_PROMPT = max(comparison, key=lambda v: average_exact(comparison[v]))
for v in comparison:
    print(f"{v}: average exact match {average_exact(comparison[v]):.1%}, {comparison[v]['avg_seconds_per_doc']} s/doc")
print("\nBest prompt:", BEST_PROMPT)

## 9. Look at the mistakes

The most useful step: see *what* goes wrong, not just how often.

In [ ]:
for field in ["company", "date", "address", "total"]:
    print(f"\n----- {field} (prompt {BEST_PROMPT}) -----")
    for ex in worst_examples(f"{OUTPUT_ROOT}/runs/train_{BEST_PROMPT}", field, n=5):
        print(f"  label:     {ex['label']}")
        print(f"  predicted: {ex['predicted']}   (fuzzy {ex['fuzzy']}, confidence {ex['confidence']})")
        print()

## 10. Final evaluation (test split, 100 receipts)

≈ 30 minutes. These are the numbers for your README and Upwork.

In [ ]:
final = run_evaluation(pipeline, load_sroie("test", limit=100), output_dir=OUTPUT_ROOT,
                       prompt_version=BEST_PROMPT, split="test")

## 11. The results

In [ ]:
from IPython.display import Markdown
display(Markdown(open(f"{OUTPUT_ROOT}/results.md").read()))

## 12. Download the result files

Upload `results.md` and `results.json` into the `eval/` folder of your GitHub repo.

In [ ]:
from google.colab import files
files.download(f"{OUTPUT_ROOT}/results.md")
files.download(f"{OUTPUT_ROOT}/results.json")